# ContractNLI 메인 실험 — RTX 5070 (Blackwell)

**Windows · PyCharm Jupyter Notebook · Python 3.11.9 · RTX 5070 약 12GB** 기준입니다.

설치 조합은 **PyTorch 2.8.0 + CUDA 12.8 wheel, bitsandbytes 0.48.1, Transformers 4.51.3**입니다. RTX 50 시리즈에서는 이전 노트북의 PyTorch 2.6.0/CUDA 12.4 설치 셀을 사용하지 마세요.

Original / Naive / Contextualized / LongLLMLingua의 4조건, 3개 정답 라벨, 원문 토큰 30% 보존 목표는 유지합니다. Qwen2.5 요약 → 기본 Llama-2-7B LongLLMLingua scorer → Qwen3 판단 순으로 한 모델씩 NF4 4bit로 로딩하고 해제합니다. 배치는 1입니다.

이 PC에서는 새 결과 폴더 `outputs/rtx5070_smoke`와 `outputs/rtx5070_base`를 사용합니다. 이전 PC의 실행 결과를 이 폴더로 복사하지 마세요. dxdiag의 전체 그래픽 메모리는 공유 RAM을 포함하며, 모델 적재에 쓰는 전용 VRAM은 약 12GB입니다.

## 0. PyCharm 준비

1. GitHub의 JJJ 저장소를 프로젝트로 엽니다.
2. PyCharm의 프로젝트 Python Interpreter에서 **Python 3.11.9 기반 가상환경**을 만듭니다. 이미 있다면 해당 환경을 선택합니다.
3. 그 환경을 선택한 PyCharm Terminal에서 필요한 경우 `python -m pip install jupyter ipykernel`을 실행합니다.
4. 이 파일을 열고 Jupyter 서버/커널의 인터프리터도 **같은 환경의 `.venv\Scripts\python.exe`**로 선택합니다. IDE 인터프리터와 노트북 커널이 다를 수 있으므로 다음 셀의 경로를 꼭 확인하세요.

노트북 셀 안에서 `python -m venv` 또는 `source`를 입력하지 않습니다. 아래 설치 셀은 IPython의 `%pip`를 사용합니다.

## 1. 커널·작업 폴더 확인

아래 셀은 최초 실행 및 커널 재시작 후 실행합니다. 저장소가 자동으로 검색되지 않으면 `PROJECT_ROOT_OVERRIDE`에 본인 PC의 경로를 입력하세요.

In [ ]:
import os
import sys
from pathlib import Path

print("Python:", sys.version)
print("Kernel executable:", sys.executable)
if sys.version_info[:3] != (3, 11, 9):
    raise RuntimeError(
        "이 노트북은 Python 3.11.9 기준입니다. PyCharm에서 해당 가상환경의 커널을 선택하세요."
    )

# 자동 검색에 실패할 때만 수정하세요. 예: r"C:\Users\YourName\PycharmProjects\JJJ"
PROJECT_ROOT_OVERRIDE = ""

if PROJECT_ROOT_OVERRIDE:
    PROJECT_ROOT = Path(PROJECT_ROOT_OVERRIDE).expanduser().resolve()
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT_ROOT = next(
        (p for p in candidates
         if (p / "configs/rtx5070.toml").is_file()
         and (p / "src/contractnli").is_dir()),
        None,
    )

if PROJECT_ROOT is None or not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise RuntimeError("PROJECT_ROOT_OVERRIDE에 git clone한 JJJ 폴더 경로를 지정하세요.")

os.chdir(PROJECT_ROOT)

# torch를 처음 import하기 전에 GPU를 한 개만 노출합니다.
if "torch" in sys.modules and os.environ.get("CUDA_VISIBLE_DEVICES") != "0":
    raise RuntimeError("커널을 재시작한 뒤 이 셀부터 다시 실행하세요.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Hugging Face를 import하기 전에 다운로드 대기 시간을 설정합니다.
os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "120")
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")
print("Project root:", PROJECT_ROOT)
print("Visible physical GPU:", os.environ["CUDA_VISIBLE_DEVICES"])

## 2. RTX 5070용 설치 — 새 가상환경에서 한 번만

아래 3개 셀을 각각 실행합니다. PyTorch는 **CUDA 12.8 공식 wheel**로 설치합니다. 프로젝트의 `rtx5070` extra는 torch 2.8.0과 bitsandbytes 0.48.1을 고정합니다. CUDA Toolkit을 따로 설치하는 명령은 아니며, 호환 NVIDIA 드라이버가 필요합니다.

설치 후 커널을 반드시 재시작하세요. 이미 이 조합을 설치했다면 설치 셀은 건너뛸 수 있습니다.

In [ ]:
%pip install --upgrade pip

In [ ]:
%pip install --upgrade torch==2.8.0 --index-url https://download.pytorch.org/whl/cu128

In [ ]:
%pip install -e ".[inference,quantization,longllmlingua,rtx5070]"

## 3. 커널 재시작

설치가 끝나면 PyCharm의 Jupyter **Restart Kernel** 기능으로 커널을 재시작하세요. 그 다음 **1번 환경 확인 코드 셀을 다시 실행**하고, 설치 셀 3개는 건너뛰어 아래 GPU 확인으로 이동합니다. 설치와 GPU 추론을 한 번의 Run All로 실행하지 마세요.

In [ ]:
import gc
import torch
import transformers
import bitsandbytes as bnb

GPU_CHECK_PASSED = False
print("PyTorch:", torch.__version__)
print("PyTorch wheel CUDA:", torch.version.cuda)
print("Transformers:", transformers.__version__)
print("bitsandbytes:", bnb.__version__)
print("CUDA available:", torch.cuda.is_available())

if not torch.cuda.is_available():
    raise RuntimeError("CUDA를 사용할 수 없습니다. 커널, CUDA wheel 설치, NVIDIA 드라이버를 확인하세요.")
if torch.cuda.device_count() != 1:
    raise RuntimeError("커널을 재시작하고 1번 셀에서 GPU 0 노출 설정을 적용하세요.")
if torch.__version__.split("+")[0] != "2.8.0" or torch.version.cuda != "12.8":
    raise RuntimeError("이 실험 프로필은 torch 2.8.0/CUDA 12.8입니다. 설치 셀 실행 후 커널을 재시작하세요.")
if bnb.__version__ != "0.48.1":
    raise RuntimeError("rtx5070 extra로 bitsandbytes 0.48.1을 설치하고 커널을 재시작하세요.")

properties = torch.cuda.get_device_properties(0)
capability = torch.cuda.get_device_capability(0)
architectures = torch.cuda.get_arch_list()
free_bytes, total_bytes = torch.cuda.mem_get_info(0)
print("GPU:", properties.name)
print("Compute capability:", capability)
print("Wheel architectures:", architectures)
print(f"VRAM total: {properties.total_memory / 1024**3:.2f} GiB")
print(f"VRAM currently free: {free_bytes / 1024**3:.2f} GiB")
if capability >= (12, 0) and "sm_120" not in architectures:
    raise RuntimeError("현재 PyTorch wheel에 Blackwell sm_120 지원이 없습니다. CUDA 12.8 설치 셀을 확인하세요.")

# CUDA available=True만으로 실제 커널 호환성을 확인할 수는 없습니다.
with torch.inference_mode():
    matrix = torch.randn(64, 64, device="cuda:0", dtype=torch.float16)
    product = matrix @ matrix.T
    torch.cuda.synchronize()
    if not torch.isfinite(product).all().item():
        raise RuntimeError("CUDA 연산 결과가 유효하지 않습니다.")
print("CUDA tensor test: PASS")

# 대형 모델을 받기 전에 Windows/Blackwell NF4 커널을 직접 확인합니다.
with torch.inference_mode():
    layer = bnb.nn.Linear4bit(
        64, 32, bias=False, compute_dtype=torch.float16,
        quant_type="nf4", compress_statistics=True,
    ).to("cuda:0")
    probe_input = torch.randn(2, 64, device="cuda:0", dtype=torch.float16)
    probe_output = layer(probe_input)
    torch.cuda.synchronize()
    if probe_output.shape != (2, 32) or not torch.isfinite(probe_output).all().item():
        raise RuntimeError("NF4 4bit 연산 결과가 유효하지 않습니다.")
print("bitsandbytes NF4 test: PASS")

del matrix, product, layer, probe_input, probe_output
gc.collect()
torch.cuda.empty_cache()
GPU_CHECK_PASSED = True
print("GPU compatibility checks: PASS. 다음 설정 셀로 진행하세요.")

## 3-1. LongLLMLingua 토큰 경계 호환 보정

LLMLingua 0.2.2의 get_prefix_length는 encode/decode 뒤 문자열이 정확히 같다고 가정합니다. 공백·문장부호 정리 등으로 이 가정이 깨지면 AssertionError가 날 수 있습니다. 아래 보정은 기존 계산을 먼저 사용하고, 그 검사가 실패한 경우에만 fast tokenizer의 원문 문자 위치로 경계를 검증합니다. 경계를 가로지르는 토큰이 있으면 중단하며 임의 위치를 반환하지 않습니다.

모델·예산·랭킹·조건 비교 설정은 그대로입니다. tools 아래 보정 파일은 실험 본체 소스와 별도로 버전과 SHA256을 run.json에 기록합니다. 기존 요약·성공한 압축·판단 결과와 원래 소스/데이터/설정 검사를 유지합니다. 커널을 재시작할 때 이 셀도 다시 실행하세요.

In [ ]:
import runpy

PREFIX_FIX_PATH = PROJECT_ROOT / "tools/llmlingua_prefix_compat.py"
if not PREFIX_FIX_PATH.is_file():
    raise FileNotFoundError("최신 코드의 tools/llmlingua_prefix_compat.py를 프로젝트에 복사하세요.")
_prefix_fix_namespace = runpy.run_path(str(PREFIX_FIX_PATH), run_name="contractnli_prefix_compat")
PREFIX_FIX_INFO = _prefix_fix_namespace["apply_prefix_fix"]()
print("LongLLMLingua prefix compatibility:", PREFIX_FIX_INFO["id"])
print("Compatibility source SHA256:", PREFIX_FIX_INFO["implementation_sha256"])

## 4. RTX 5070 실험 설정

`configs/rtx5070.toml`을 사용합니다. GPU 호환 확인 셀이 통과해야 진행합니다. 모델 크기·프롬프트·LongLLMLingua 설정·라벨·30% 목표는 기존 base 실험과 같습니다. 전체 실험 출력은 이 PC 전용 새 폴더에 저장합니다.

In [ ]:
if not globals().get("GPU_CHECK_PASSED", False):
    raise RuntimeError("CUDA tensor test와 NF4 test가 PASS인 것을 먼저 확인하세요.")

import json
from dataclasses import replace

from contractnli.config import load_config
from contractnli.data import load_dataset
from contractnli.download import download
from contractnli.experiment import Experiment, inspect
from contractnli.preflight import preflight

BASE_CONFIG = load_config(PROJECT_ROOT / "configs/rtx5070.toml")
assert all(
    model.device == "cuda" and model.quantization == "4bit" and model.dtype == "float16"
    for model in (BASE_CONFIG.summarizer, BASE_CONFIG.compressor, BASE_CONFIG.judge)
), "단일 GPU 4bit 설정을 확인하세요."
assert len(BASE_CONFIG.conditions) == 4
assert BASE_CONFIG.retention_ratio == 0.30
print(json.dumps(BASE_CONFIG.to_dict(), indent=2, ensure_ascii=False))

## 5. 데이터 준비·평가 대상 확인

공식 ZIP을 고정 commit에서 받아 체크섬을 확인합니다. 이미 데이터가 있으면 재사용합니다. NotMentioned를 포함한 **123문서, 2,091문제**를 유지합니다. Gold evidence는 1,188문제에만 존재합니다.

In [ ]:
DATA_PATH = PROJECT_ROOT / BASE_CONFIG.dataset
if not DATA_PATH.is_file():
    download(DATA_PATH.parent)
else:
    print("기존 데이터 재사용:", DATA_PATH)

dataset = load_dataset(DATA_PATH)
inventory = inspect(BASE_CONFIG, dataset)
print(json.dumps(inventory, indent=2, ensure_ascii=False))
if inventory["documents"] != 123 or inventory["evaluated_pairs"] != 2091:
    raise RuntimeError("공식 test split 123문서/2,091문제가 아닙니다. dataset 경로를 확인하세요.")

## 6. 입력 길이 사전 검증

토크나이저만 로딩하고 모델 가중치는 로딩하지 않습니다. 원문+출력 예산이 모델 입력 한도를 넘는 경우 중단합니다. 이 단계가 통과해도 GPU OOM이 없다는 보장은 없습니다.

In [ ]:
length_check = preflight(BASE_CONFIG)
print(json.dumps(length_check, indent=2, ensure_ascii=False))
if not length_check["ok"]:
    raise RuntimeError("입력 길이 한도 초과가 있습니다. 원문을 임의로 자르지 말고 초과 항목을 확인하세요.")

## 7. 한 문서로 GPU 실행 확인

처음에는 한 문서의 17가설 × 4조건 = **68개 판단**을 실행합니다. 요약·압축 생성 시간도 들어가므로 즉시 끝나는 테스트는 아닙니다. 작은 실험과 전체 실험의 출력 폴더를 분리합니다.

실행 중 `summary`, `compression`, `judgment` 진행 로그가 표시됩니다. 최초 실행에는 큰 모델 파일 다운로드도 포함됩니다. 다운로드 표시가 100%가 되어도 요약·압축·판단은 계속 진행될 수 있습니다. **중지 버튼을 누르지 말고 `Smoke run completed: True`가 출력되고 셀 실행이 끝날 때까지 기다리세요.** 그 후에만 8번 셀을 수정하고 실행합니다.

실수로 중지했다면 같은 셀을 다시 실행하세요. 완료한 항목과 다운로드 캐시는 재사용합니다. 현재 출력 폴더 이름의 `rtx5070`은 기존 결과와의 호환을 위해 유지한 이름이며 실제 GPU를 제한하지 않습니다. 커널 강제 종료 시에는 README의 잠금 복구 절차를 확인하세요.

In [ ]:
SMOKE_DOCUMENTS = 1
SMOKE_CONFIG = replace(
    BASE_CONFIG,
    max_documents=SMOKE_DOCUMENTS,
    output_dir="outputs/rtx5070_smoke",
)
# 재실행을 중단했을 때 이전 성공 결과가 남아 있지 않도록 초기화합니다.
smoke_report = None
print("한 문서 검증을 실행합니다. 완료 메시지가 나올 때까지 중지하지 마세요.", flush=True)
smoke_report = Experiment(SMOKE_CONFIG).run("all")
print("Smoke run completed:", smoke_report["complete"])
print(f"현재 PyTorch GPU allocated: {torch.cuda.memory_allocated(0) / 1024**3:.3f} GiB")
print("이 결과는 실행 확인용 부분집합이며 전체 test 성능이 아닙니다.")

## 8. 전체 메인 실험

**7번 셀의 `Smoke run completed: True` 출력과 셀 실행 종료를 확인한 뒤**, 아래 `RUN_FULL_EXPERIMENT = False`를 `True`로 바꾸고 **8번 셀만 실행**하세요. 7번 셀을 중지해서 다음 단계로 넘어가는 것이 아닙니다. GPU는 현재 PC에서 자동 인식한 장치 한 개를 사용합니다.

1. Qwen2.5를 올려 Naive·Contextualized 문서 요약 246회를 수행하고 해제합니다.
2. 기본 Llama-2-7B scorer를 올려 LongLLMLingua 가설별 압축 2,091회를 수행하고 해제합니다.
3. Qwen3를 올려 네 문맥 조건의 판단 8,364회를 수행하고 해제합니다.
4. 최종 지표를 저장합니다.

다른 노트북이나 프로세스에서 모델을 함께 올리지 마세요. 같은 코드를 재실행하면 완료된 결과를 재사용합니다. 실험 중 설정·모델 revision·소스·환경을 바꾸면 이전 체크포인트 재사용을 거부합니다.

이 PC의 smoke run이 성공한 뒤 이 PC에서 본 실험을 처음부터 진행합니다. 다른 PC의 체크포인트·처리시간을 이어 붙이지 않습니다.

In [ ]:
RUN_FULL_EXPERIMENT = False

if RUN_FULL_EXPERIMENT:
    if not isinstance(globals().get("smoke_report"), dict) or smoke_report.get("complete") is not True:
        raise RuntimeError(
            "7번 smoke 셀이 아직 완료되지 않았습니다. 중지하지 말고 끝까지 실행하여 "
            "Smoke run completed: True 메시지를 확인한 뒤 8번 셀을 실행하세요."
        )
    full_report = Experiment(BASE_CONFIG).run("all")
    print("전체 실험 완료:", full_report["complete"])
    print("결과 폴더:", (PROJECT_ROOT / BASE_CONFIG.output_dir).resolve())
else:
    print("7번 셀의 Smoke run completed: True와 실행 종료를 확인한 다음, 이 셀의 값을 True로 바꾸세요.")

## 9. 결과 확인

전체 실행을 선택했으면 전체 결과를, 그렇지 않으면 smoke 결과를 표시합니다. 수치는 0–1 비율입니다. `metrics.csv`는 Excel에서도 열 수 있습니다.

In [ ]:
from IPython.display import Markdown, display

result_directory = PROJECT_ROOT / (
    BASE_CONFIG.output_dir if RUN_FULL_EXPERIMENT else SMOKE_CONFIG.output_dir
)
metrics_path = result_directory / "metrics.json"

if metrics_path.is_file():
    report = json.loads(metrics_path.read_text(encoding="utf-8"))
    print("부분집합 결과:", report["dataset"]["partial_dataset"])
    print("평가 문항 수:", report["dataset"]["evaluated_pairs"])
    headers = ["조건", "Accuracy", "Macro-F1", "Decision flip", "토큰 보존 비율"]
    table = ["| " + " | ".join(headers) + " |", "|---|---|---|---|---|"]
    for condition, metrics in report["metrics"].items():
        values = [metrics["accuracy"], metrics["macro_f1"],
                  metrics.get("decision_flip_rate"), metrics["mean_actual_token_retention"]]
        formatted = ["—" if value is None else f"{value:.4f}" for value in values]
        table.append("| " + " | ".join([condition, *formatted]) + " |")
    display(Markdown("\n".join(table)))
    print("상세 JSON:", metrics_path)
    print("CSV:", result_directory / "metrics.csv")
else:
    print("아직 모든 예측이 완료되지 않았습니다:", result_directory)

## 중단·재개와 메모리 부족

- 일반적인 셀 Interrupt 후에는 같은 smoke/full 셀을 다시 실행하세요. 완료된 항목을 건너뜁니다.
- GPU OOM이 발생하면 다른 GPU 프로그램·노트북의 추론을 종료하고 커널을 재시작한 뒤, 1번 → GPU 확인 → 설정·데이터 → 해당 실행 셀 순으로 진행하세요. 설치 셀은 다시 실행할 필요가 없습니다.
- 커널이 강제로 종료되어 `.run.lock`이 남았다면 **해당 실험 프로세스가 종료된 것을 확인한 후에만** 해당 출력 폴더의 잠금 파일을 삭제하세요. `contexts.jsonl`, `predictions.jsonl`, `run.json`은 보존하세요.
- 같은 문서가 계속 OOM이면 GPU 이름·실제 VRAM·오류·실패 문서 ID를 기록해야 합니다. 입력을 조용히 자르거나 일부 문항을 건너뛰어 전체 실험이 성공한 것처럼 보고하지 않습니다.
- 설치·환경 변경이 필요한 경우 기존 결과를 보존하고 새 출력 폴더를 사용하세요. 이미 실행한 결과를 지우는 명령은 이 노트북에 없습니다.

이 노트북은 Windows/PyCharm 사용을 위해 작성되었습니다. 3060·3070 등 실제 GPU의 이름과 메모리는 자동으로 표시됩니다. GPU가 바뀌면 새 장치에서도 한 문서 검증부터 진행해야 합니다. 전체 모델 가중치를 사용한 실행 검증은 사용자의 GPU 환경에서 수행해야 합니다.

## 설치 조합의 공식 근거
[PyTorch 공식 CUDA 12.8 wheel 목록](https://download.pytorch.org/whl/cu128/torch/)에는 Python 3.11/Windows용 torch 2.8.0 wheel이 있습니다.
[PyTorch 2.8.0 공식 빌드 설정](https://github.com/pytorch/pytorch/blob/v2.8.0/.ci/manywheel/build_cuda.sh)은 CUDA 12.8에서 compute capability 12.0을 포함합니다.
[bitsandbytes 0.48.1 공식 설치 문서](https://github.com/bitsandbytes-foundation/bitsandbytes/blob/0.48.1/docs/source/installation.mdx)는 Windows x86-64/CUDA 12.8에서 sm120을 지원합니다.
이 문서 확인은 실제 사용자의 RTX 5070에서 전체 모델 추론을 검증했다는 뜻은 아닙니다. 위의 실제 CUDA·NF4 검사와 한 문서 smoke run으로 확인하세요.